In [1]:
import ollama
import json
from Error import Error
from FeedbackOutput import FeedbackOutput
from pydantic import BaseModel

# json schema (format) for get error LLM output 
class ErrorGetterFormat(BaseModel):
        error_type: str
        severity: str
        description: str

# json schema (format) for a list of get error LLM outputs 
class ErrorListFormat(BaseModel):
        errors: list[ErrorGetterFormat]

# json schema (format) for get suggestion LLM output
class SuggestionGetterFormat(BaseModel):
        suggestion: str

# json schema (format) for a list of get suggestion LLM output 
class SuggestionListFormat(BaseModel):
        suggestions: list[SuggestionGetterFormat]

prompt_config_path = "prompts.json" # path to prompts json file
host = "http://localhost:11434" # Ollama binds to port 11434 by default
model = "llama3:latest" # model name

available_error_types = "OVERFLOW_ERROR, ROUND_OFF_ERROR, INFINITE_LOOP_ERROR," \
                "MODIFY_PARAMETER_VARIABLE_ERROR, OFF_BY_ONE_ERROR, ARITHMETIC_ERROR," \
                "BOUNDS_ERROR, UNINITIALIZED_ARRAY_ERROR, SQUELCH_EXCEPTION_ERROR," \
                "MAGIC_NUMBER_ERROR, DANGLING_ELSE_ERROR, OTHERS"

# input from the changes in mined repo's pull request
code = "" \
        "counter = 0:\n" \
        "while counter < 5:\n" \
        "\tprint(counter)" 

In [2]:
# get the prompts from the json file

try:
    with open(prompt_config_path, "r", encoding="utf-8") as file:
        prompts = json.load(file)

        system_prompt_error = prompts['system_prompt_error']
        print(f"System prompt error fetched: {system_prompt_error}")

        system_prompt_suggestion = prompts['system_prompt_suggestion']
        print(f"System prompt suggestion fetched: {system_prompt_suggestion}")

        get_error_prompt = prompts['get_error_prompt']
        print(f"Get error prompt fetched: {get_error_prompt}")

        suggestion_prompt = prompts['suggestion_prompt']
        print(f"Suggestion prompt fetched: {suggestion_prompt}")

        print(f"\n'{prompt_config_path}' successfully loaded!")
        
except FileNotFoundError:
    print(f"Error: The file '{prompt_config_path}' was not found.")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}")

client = ollama.Client(host=host)

System prompt error fetched: You are an expert software engineer doing python code review. When given a code sample and a list of possible errors, find if any of the errors exist in the code. You can only choose up to 2 error types. Always return your response as a Python list of strings in this format: Error Type | Error severity (low/medium/high) | Error description. If no errors are found return: NONE
System prompt suggestion fetched: You are an expert software engineer doing python code review. Suggest a fix for an error when you are given an error type, error severity level, and error description. You should only write one suggestion
Get error prompt fetched: Here is the code sample {code} and the list of possible errors {available_error_types}
Suggestion prompt fetched: Here is the error type: {error_type}, severity level: {severity} and description: {error_description}. This is the code snippet: {code}.

'prompts.json' successfully loaded!


In [ ]:
# get error prompt

def request_error(get_error_prompt: str, code: str) -> str:
    """
    use LLM to check if there are any errors in the code.
    Will return a json-like string.
    """
    get_error_prompt = get_error_prompt.format(
        code = code, # the code in string format containing the changes in a specific pull request mined from the repo
        available_error_types = available_error_types
    )

    get_error_response = client.chat(
        model = model,
        messages = [
            {"role": "system", "content": system_prompt_error},
            {"role": "user", "content": get_error_prompt}
        ],
        format = ErrorListFormat.model_json_schema()
    )

    error_response = get_error_response["message"]["content"]
    return error_response

def parse_error_response(error_response: str, code: str) -> list[Error]:
    """
    parse the json-like string into a json object.
    Then create Error objects containing all the information
    """   
    if error_response == "NONE":
        return []
    
    error_response = json.loads(error_response)
    
    parsed_errors = error_response['errors'] # end result or parsing the error response from LLM: a list of error information in consistent format

    # Create a list of Error objects. Assign error type, severity level, and error description generated from llm
    errors = []

    for idx, error_info in enumerate(parsed_errors):
        error_type = error_info['error_type']
        error_severity_level = error_info['severity']
        error_description = error_info['description']

        error = Error(error_type, error_severity_level, error_description, code)
        errors.append(error)

    return errors

error_response = request_error(get_error_prompt, code)
errors = parse_error_response(error_response, code)

for error in errors:
    print(error.error_type)
    print(error.error_severity_level)
    print(error.error_description)
    print()

INFINITE_LOOP_ERROR
high
The while loop will run indefinitely because the counter is not being incremented.

BOUNDS_ERROR
medium
The while loop condition may never be met, causing the loop to potentially run infinitely.



In [ ]:
# get fix suggestion

def request_suggestion(error: Error) -> Error:
    """
    use LLM to give suggestions to all the errors.
    Will return an Error object.
    """
    error_type = error.get_error_type()
    severity = error.get_error_severity_level()
    error_description = error.get_error_description()
    code = error.get_code()

    global suggestion_prompt

    suggestion_prompt = suggestion_prompt.format(
        error_type = error_type,
        severity = severity,
        error_description = error_description,
        code = code
    )

    suggestion_response = client.chat(
        model = model,
        messages = [
            {"role": "system", "content": system_prompt_suggestion},
            {"role": "user", "content": suggestion_prompt}
        ],
        format=SuggestionListFormat.model_json_schema()
    )

    suggestion_response = suggestion_response["message"]["content"]

    suggestion_response_parsed = json.loads(suggestion_response)['suggestions'][0]['suggestion']

    error.set_fix_suggestion(suggestion_response_parsed)

    return error

def get_all_fix_suggestions(errors: list[Error]) -> list[Error]:
    """
    iterates through every Error and ask the LLM fix suggestions for each of them.
    Returns a list of Error objects.
    """
    for idx, error in enumerate(errors):
        errors[idx] = request_suggestion(error)

    return errors

errors = get_all_fix_suggestions(errors)

feedback_output = FeedbackOutput(errors)

print(feedback_output)

Timestamp: 2026-03-25 08:34:21
-------------------------
Error type: INFINITE_LOOP_ERROR
Severity: high
Description: The while loop will run indefinitely because the counter is not being incremented.
Suggestion: You need to increment the counter variable inside the loop. Here's how you can do it:

```
counter = 0
while counter < 5:
    counter += 1
    print(counter)
```
-------------------------
Error type: BOUNDS_ERROR
Severity: medium
Description: The while loop condition may never be met, causing the loop to potentially run infinitely.
Suggestion: Add an increment operator to increase the counter value inside the while loop. This will ensure the loop condition is met and the program terminates properly. For example, change `counter = 0: while counter < 5:` to `counter = 0; i = 0: while i < 5:`.

